In [52]:
# imports

import pandas as pd
import numpy as np
from dash import Dash, html, dcc, callback, Output, Input
import dash_ag_grid as dag
import plotly.express as px

In [61]:
# import patient data

df = pd.read_csv("./patients.csv")

df['last_visit'] = pd.to_datetime(df['last_visit'], format='%m/%d/%y').dt.date
df['followup'] = pd.to_datetime(df['followup'], format='%m/%d/%y').dt.date
df['discharged/deceased'] = pd.to_datetime(df['discharged/deceased'], format='%m/%d/%y').dt.date
df['active'] = df['discharged/deceased'].isna().astype(int)
df['first_name'] = df['first_name'].str.capitalize()
df['last_name'] = df['last_name'].str.capitalize()

# changing dates to datetime format; checks actually works on first row

print(type(df["last_visit"][0]))
df['last_visit'] = pd.to_datetime(df['last_visit'], format='%m/%d/%y')
print(type(df["last_visit"][0]))

print(type(df["followup"][0]))
df['followup'] = pd.to_datetime(df['followup'], format='%m/%d/%y')
print(type(df["followup"][0]))

# app with only data in a table

# Initialize the app
app = Dash()

# App layout
app.layout = [
    html.Div(children='Basic Patient Data'),
    dag.AgGrid(
        rowData=df.to_dict('records'),
        columnDefs=[{"field": i} for i in df.columns]
    )
]

# Run the app
if __name__ == '__main__':
    app.run(debug=True)

# app with table and histogram

# Initialize the app
app = Dash()

# App layout
app.layout = [
    html.Div(children='Basic Patient Data'),
    dag.AgGrid(
        rowData=df.to_dict('records'),
        columnDefs=[{"field": i} for i in df.columns]
    ),
    dcc.Graph(figure=px.histogram(df, x='dx', y='dx', histfunc='count'))
]

# dcc.Graph(figure=px.histogram(df, x='continent', y='lifeExp', histfunc='sum'))

# Run the app
if __name__ == '__main__':
    app.run(debug=True)

In [ ]:
# INTERACTIVE app with table and histogram

# Initialize the app
app = Dash()

# App layout
app.layout = [
    html.Div(children='Basic Patient Data'),
    html.Hr(),
    dcc.RadioItems(options=['dx', 'last_name', 'sex'], value='dx', id='controls-and-radio-item'),
    dag.AgGrid(
        rowData=df.to_dict('records'),
        columnDefs=[{"field": i} for i in df.columns]
    ),
    dcc.Graph(figure={}, id='current_graph'),
    dcc.Dropdown(value=[], id = 'drop_down_box')
]

# Add controls to build the interaction
@callback(
    Output(component_id='current_graph', component_property = 'figure'),
    # Output(component_id='drop_down_box', component_property = 'value'),      ##### trying to get this to fill in a dropdown box, so I can select "anxiety" and get a count for number of patients; then later add one for "due for followup", choose a date, and show everything after that date.
    Input(component_id='controls-and-radio-item', component_property = 'value')
)
def update_graph(col_chosen):
    ordered_axis = sorted(df[col_chosen].unique())
    fig = px.histogram(df, x = col_chosen, y = col_chosen, histfunc = 'count', category_orders = {col_chosen : ordered_axis}, text_auto = True)
    fig.update_layout(yaxis_title = f"{col_chosen} Total")
    return fig



# Run the app
if __name__ == '__main__':
    app.run(debug=True)

In [27]:
# INTERACTIVE app with table and histogram

# Initialize the app
app = Dash()

# App layout
app.layout = ([
    html.Div(children='Basic Patient Data'),
    html.Hr(),
    dcc.RadioItems(options=['dx', 'last_name', 'sex'], value='dx', id='controls-and-radio-item'),
    html.Hr(),
    dag.AgGrid(
        rowData=df.to_dict('records'),
        columnDefs=[{"field": i} for i in df.columns]
    ),
    html.Hr(),
    dcc.Graph(figure={}, id='current_graph'),
    html.Hr(),
    dcc.Dropdown(
    options=sorted(df['dx'].unique()),
    value=None,
    id='patients_with_condition'
    ),
    # dcc.Dropdown(value=[], options = sorted(df['dx'].unique()), id = 'patients_with_condition'),
    html.Hr(),
    dag.AgGrid(
        # ldf = df[df['dx']==f'listed_condition'][['first_name','last_name']].copy(),
        rowData = [],
        columnDefs=[{"field": i} for i in df.columns],
        id = 'condition_patient_table'
    )
])

# Add controls to build the interaction
@callback(
    Output(component_id='current_graph', component_property = 'figure'),
    Input(component_id='controls-and-radio-item', component_property = 'value')
)
def update_graph(col_chosen):
    ordered_axis = sorted(df[col_chosen].unique())
    fig = px.histogram(df, x = col_chosen, y = col_chosen, histfunc = 'count', category_orders = {col_chosen : ordered_axis}, text_auto = True)
    fig.update_layout(yaxis_title = f"{col_chosen} Total")
    return fig

@callback(
    Output(component_id = 'condition_patient_table', component_property= 'rowData'),
    Input(component_id = 'patients_with_condition', component_property = 'value')
)
def update_patients_with_condition_graph(value):
    ldf = df[df['dx'] == value][['first_name','last_name']].copy()
    return ldf.to_dict('records')

# Run the app
if __name__ == '__main__':
    app.run(debug=True)

In [31]:
# expanded interactive version
app = Dash()

app.layout = html.Div([
    html.Div(children='Basic Patient Data'),
    html.Hr(),

    dcc.RadioItems(
        options=['dx', 'last_name', 'sex'],
        value='dx',
        id='controls-and-radio-item'
    ),

    html.Hr(),

    dag.AgGrid(
        rowData=df.to_dict('records'),
        columnDefs=[{"field": i} for i in df.columns]
    ),

    html.Hr(),

    dcc.Graph(id='current_graph'),

    html.Hr(),

    dcc.Dropdown(
        options=sorted(df['dx'].unique()),
        value=None,
        id='patients_with_condition'
    ),

    html.Hr(),

    dag.AgGrid(
        rowData=[],
        columnDefs=[
            {"field": "first_name"},
            {"field": "last_name"}
        ],
        id='condition_patient_table'
    )
])

# Histogram callback
@callback(
    Output('current_graph', 'figure'),
    Input('controls-and-radio-item', 'value')
)
def update_graph(col_chosen):
    ordered_axis = sorted(df[col_chosen].unique())
    fig = px.histogram(
        df,
        x=col_chosen,
        histfunc='count',
        category_orders={col_chosen: ordered_axis},
        text_auto=True
    )
    fig.update_layout(yaxis_title=f"{col_chosen} Total")
    return fig

# Patient table callback
@callback(
    Output('condition_patient_table', 'rowData'),
    Input('patients_with_condition', 'value')
)
def update_patients_with_condition_graph(value):
    if value is None:
        return []

    ldf = df[df['dx'] == value][['first_name', 'last_name']]
    return ldf.to_dict('records')

# Run app
if __name__ == '__main__':
    app.run(debug=True)


In [65]:
# check out only count by total ACTIVE cases

app = Dash()

app.layout = html.Div([
    html.Div(children='Total Active Cases'),

    html.Hr(),

    dcc.Graph(
        fig = px.histogram(
        df[df['active'] == 1],
        histfunc='count',
        text_auto=True
    ))
])

# Run app
if __name__ == '__main__':
    app.run(debug=True)


ValueError: Plotly Express cannot process wide-form data with columns of different type.

In [62]:
df

,id,first_name,last_name,sex,dx,last_visit,followup,discharged/deceased,deceased,deceased/medical reason,active
0,322,Joe,Marx,m,sick,2025-01-01,2026-05-01,NaT,n,NaN,1
1,326,Jane,Doe,f,cholera,2025-01-02,2026-05-02,NaT,n,NaN,1
2,975,Johnny,Cash,m,alcoholism,2025-01-11,2026-05-03,NaT,n,NaN,1
3,1278,Suzy,Hudson,f,pneumonia,2025-01-04,2026-05-04,NaT,n,NaN,1
4,599,Sam,Seuss,m,psychosis,2025-01-05,2026-05-05,NaT,n,NaN,1
5,370,Susan,Stevens,f,vocal injury,2025-01-19,2026-05-06,NaT,n,NaN,1
6,398,Hudson,Williams,m,sick,2025-01-09,2026-05-07,NaT,n,NaN,1
7,307,Connor,Storie,m,very hot,2025-01-08,2026-05-08,NaT,n,NaN,1
8,1053,Shane,Hollander,m,autism,2025-01-09,2026-05-09,NaT,n,NaN,1
9,386,Ilya,Razanov,m,very hot,2025-01-08,2026-05-10,NaT,n,NaN,1


In [ ]:
df['discharged/deceased'][0] == np.isnan

TypeError: isnan() takes from 1 to 2 positional arguments but 0 were given